In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("ps05.ipynb")

# PS5 — Feedforward Loops and Oscillation
### BioE 147/247 · Fall 2026

**Out:** Sunday, September 27 · **Due:** Thursday, October 8, 11:59 pm
**Covers:** Sessions 10 and 11 · **42 points** · **Question 6** is required for **BioE 247** (48 total) and is extra credit for **BioE 147** (up to +6)

---

Two sessions, two circuits, one question: what does a circuit do with *time*?
Session 10's feedforward loop delays one event or turns a step into a pulse;
session 11's ring of three refuses to settle at all. Questions 1–3 are the
first, 4–5 the second, and Q6 joins them.

Two of these questions are items your handouts said would be here. They are the
same items.

**The delay convention**, which is printed on the session 10 handout and which
you must use throughout:

> The **response time** is the time for $Z$ to reach **50% of its steady-state
> value**. A **delay** is measured only against a comparison circuit — simple
> regulation of $Z$ by $X$ alone, with $\beta_z$ chosen so that both designs
> reach the **same steady-state $Z$**. Then
> $t_D = t_{1/2}(\text{FFL}) - t_{1/2}(\text{simple})$, reported
> separately for the ON and the OFF step.

A delay reported without its comparison and its direction is not a number
anyone else can check, and it earns no marks here.

**Collaboration is encouraged.** Discuss, argue, work at a whiteboard together,
then write your own solution and your own code. Record who you worked with
below.

**If you used an LLM**, say so briefly and say what for. The conditions are that
you can explain anything you submit and that the code you submit runs.

**A note on the visible tests.** They check *properties* — symmetries, counts,
what happens at a limit. A green visible check means "not obviously broken",
not "right".

In [ ]:
COLLABORATORS = ""   # e.g. "worked with J. Chen on Q3"
AI_USE = ""          # e.g. "used an LLM to debug the bisection in Q5"

## Setup

In [ ]:
# ---------------------------------------------------------------------------
# SETUP — run this cell first, every time.
#
# DataHub / local : finds the repository root and puts it on the import path.
# Google Colab    : clones the repository first, because Colab opens this
#                   notebook on its own, without the posb package beside it.
# ---------------------------------------------------------------------------
import os
import sys

if "google.colab" in sys.modules:
    if not os.path.exists("posb2026"):
        !git clone -q https://github.com/ArkinLaboratory/posb2026.git
    sys.path.insert(0, os.path.abspath("posb2026"))
else:
    _d = os.getcwd()
    while _d != os.path.dirname(_d) and not os.path.isdir(os.path.join(_d, "posb")):
        _d = os.path.dirname(_d)
    sys.path.insert(0, _d)

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

import posb
from posb import Reaction, Model

posb.check_environment()


In [ ]:
from posb import (Model, Reaction, fixed_points, jacobian, classify,
                  stability_report, repressilator_model,
                  repressilator_alpha_critical)
from scipy.optimize import brentq

### The feedforward loop, given

You built this circuit's argument in class; here is the model, so that
Questions 1–3 are about *measuring* it rather than retyping it.

$$\frac{dY}{dt} = B_y + f(X, K_{xy}) - \alpha_y Y \qquad
\frac{dZ}{dt} = B_z + \beta_z\, G(X, Y) - Z$$

with $f(u,K) = (u/K)^H / (1 + (u/K)^H)$ for an activator and
$1/(1+(u/K)^H)$ for a repressor, and $G$ the AND gate — the product of what
each regulator contributes. $X$ is an inert species: it holds whatever the
initial condition sets, which is how a step is imposed.

Defaults everywhere below: $H = 2$, $K_{xy} = K_{xz} = 0.1$,
$\alpha_y = \beta_z = 1$, $B_y = B_z = 0$.

In [ ]:
H = 2
KXY = KXZ = 0.1


def act(u, K, H=None):
    H = globals()["H"] if H is None else H
    u = max(u, 0.0)
    return (u / K) ** H / (1 + (u / K) ** H)


def rep(u, K, H=None):
    H = globals()["H"] if H is None else H
    return 1.0 / (1 + (max(u, 0.0) / K) ** H)


def ffl(kind, Kyz, By=0.0, beta_z=1.0, alpha_y=1.0, H=None):
    """kind = "C1" (coherent type 1) or "I1" (incoherent type 1), AND gate."""
    gate = ((lambda c: act(c["X"], KXZ, H) * act(c["Y"], Kyz, H)) if kind == "C1"
            else (lambda c: act(c["X"], KXZ, H) * rep(c["Y"], Kyz, H)))
    return Model(
        [Reaction({}, {"Y": 1}, rate=lambda c, p: By + act(c["X"], KXY, H), name="Y prod"),
         Reaction({"Y": 1}, {}, k=alpha_y, name="Y removal"),
         Reaction({}, {"Z": 1}, rate=lambda c, p, g=gate: beta_z * g(c), name="Z prod"),
         Reaction({"Z": 1}, {}, k=1.0, name="Z removal")],
        species=["X", "Y", "Z"])


def simple_reg(beta_z, H=None):
    """Simple regulation of Z by X alone -- the comparison circuit."""
    return Model(
        [Reaction({}, {"Z": 1}, rate=lambda c, p: beta_z * act(c["X"], KXZ, H), name="Z prod"),
         Reaction({"Z": 1}, {}, k=1.0, name="Z removal")],
        species=["X", "Z"])


def step_response(model, x_pre, x_post, t=25.0, n_points=6000):
    """Steady state at X = x_pre, then the trajectory after X switches."""
    pre = model.steady_state({s: 0.0 for s in model.species} | {"X": x_pre})
    return model.simulate(dict(pre) | {"X": x_post}, (0.0, t), n_points=n_points)

---
## Question 1 — Read the wiring

*T24. Session 10.*

Eight ways to wire two regulators and one output, and they do not do the same
things. The rule is the one from class: compare the sign of the direct path
$X \to Z$ with the **product** of the signs along $X \to Y \to Z$.

**Q1a.** Write `coherence(s_xy, s_yz, s_xz)` taking three signs, each
`+1` or `-1`, and returning the string `"coherent"` or `"incoherent"`.

In [ ]:
def coherence(s_xy, s_yz, s_xz):
    """"coherent" if the direct and indirect paths agree in sign."""
    ...


for signs in [(1, 1, 1), (1, -1, 1), (-1, -1, 1), (-1, 1, 1)]:
    print(signs, "->", coherence(*signs))

In [ ]:
grader.check("q1a")

<!-- BEGIN QUESTION -->

**Q1b.** For each of the two circuits below, say in one sentence what it
does to a **step ON** of $X$, with an AND gate at the $Z$ promoter, and name
the one engineering job it is good for.

1. $X \xrightarrow{+} Y$, $Y \xrightarrow{+} Z$, $X \xrightarrow{+} Z$
2. $X \xrightarrow{+} Y$, $Y \xrightarrow{-} Z$, $X \xrightarrow{+} Z$

Then: one of these two is what Basu et al. built. Which, and how can you tell
from the *behavior* they reported rather than from the wiring diagram?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 2 — The delay, measured

*T25. Session 10, handout item 2.*

This is the item your handout promised. Use the convention quoted at the top of
this notebook — and state it in your own words in Q2d, because a table with no
convention is not gradeable.

**Q2a.** Write `response_time(traj)`: given a `Trajectory`, the time at
which $Z$ first reaches 50% of the way from its initial value to its final
value. Interpolate linearly between the two straddling samples — do not just
return the nearest sample time.

Then `delay(Kyz, direction)`, returning $t_D$ for the coherent type-1 AND
circuit at that $K_{yz}$, for `direction` either `"on"` or `"off"`. Build the
matched comparison circuit yourself: choose its $\beta_z$ so that it reaches the
**same** steady-state $Z$ as the FFL.

In [ ]:
def response_time(traj):
    """Time for Z to cover half the distance from its first to its last value."""
    ...


def delay(Kyz, direction="on"):
    """t_D = t_half(FFL) - t_half(matched simple regulation)."""
    ...


for K in (0.2, 0.5, 0.8):
    print(f"Kyz = {K}:  ON {delay(K, 'on'):+.3f}   OFF {delay(K, 'off'):+.3f}")

In [ ]:
grader.check("q2a")

**Q2b.** Write `delay_formula(Kyz, Ymax=1.0, Ymin=0.0, alpha_y=1.0)`,
the sharp-gate prediction derived in class:

$$t_D = \alpha_y^{-1}\ln\!\left[\frac{Y_{\max}-Y_{\min}}{Y_{\max}-K_{yz}}\right]$$

In [ ]:
def delay_formula(Kyz, Ymax=1.0, Ymin=0.0, alpha_y=1.0):
    """The H >> 1 prediction for the ON-step delay."""
    ...


for K in (0.2, 0.5, 0.8):
    print(f"Kyz = {K}:  formula {delay_formula(K):.3f}   measured {delay(K):.3f}")

In [ ]:
grader.check("q2b")

**Q2c.** The formula assumes a sharp gate. Test that claim: write
`delay_at_H(Kyz, Hval)` which recomputes the measured ON-step delay with the
Hill coefficient set to `Hval` everywhere, and show that as $H$ grows the
measurement converges on `delay_formula`.

Report `delay_at_H(0.8, H)` for $H = 2, 4, 6, 10, 20, 40$ against the formula's
1.61.

**The approach is not monotone, and that is not a bug in your code.** The
measurement rises past 1.61 somewhere around $H = 10$ before settling back.
Add one sentence saying why a gate that is merely *sharper* can make the wait
both shorter and longer than the sharp-gate answer — it is the same fact Q2b
made you notice at two different $K_{yz}$.

In [ ]:
def delay_at_H(Kyz, Hval):
    """The measured ON-step delay with every Hill coefficient set to Hval."""
    ...


print(f"formula: {delay_formula(0.8):.3f}")
for Hv in (2, 4, 10, 40):
    print(f"  H = {Hv:>2}: {delay_at_H(0.8, Hv):.3f}")

In [ ]:
grader.check("q2c")

<!-- BEGIN QUESTION -->

**Q2d.** Two short answers.

1. State the delay convention in your own words — what is measured, against
   what, and in which direction.
2. At $K_{yz} = 0.8$ the formula says 1.61 and the simulation says 1.02; at
   $K_{yz} = 0.2$ the formula says 0.23 and the simulation says 0.35. The
   formula is wrong in **opposite directions** in the two rows. Given what Q2c
   showed, say what the single underlying cause is, and why a designer should
   care.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 3 — Adaptation, measured not solved

*T26. Session 10, handout item 3.*

The incoherent type 1 with a **basal** level of $Y$: $Z$ overshoots and then
settles to a plateau that is not zero. There is no formula to plug into here.
Simulate, find the highest point, find where it settles, divide.

**Q3a.** Write `adaptation(By, Kyz=0.5, beta_z=6.0)` returning the tuple
`(peak, final, error)` for a step of $X$ from 0 to 1, where `error` is the
**adaptation error** $Z_{\text{final}} / Z_{\text{peak}}$.

Integrate long enough that `final` really is the plateau and not a point still
on its way down.

In [ ]:
def adaptation(By, Kyz=0.5, beta_z=6.0):
    """(peak, final, adaptation error) for the I1-FFL with basal Y."""
    ...


for By in (0.2, 0.4, 0.8):
    p, f, e = adaptation(By)
    print(f"By = {By}:  peak {p:.4f}  final {f:.4f}  adaptation error {e:.4f}")

In [ ]:
grader.check("q3a")

**Q3b.** A designer wants the circuit to *forget* the step as completely
as possible — an adaptation error as close to zero as they can get. The obvious
knob is $B_y$, the basal level of the repressor arm.

Sweep $B_y$ from 0.05 to 1.0 and plot the adaptation error against it.

**a.** You will find there is no interior optimum. Set `BY_AT_070` to the value
of $B_y$ at which the adaptation error first reaches $0.70$ — find it by
bisection or by interpolating your sweep, not by reading it off the plot.

**b.** In three or four sentences: which way does the error run with $B_y$, what
is the mechanism, and — since the sweep points one way — what stops you from
taking $B_y$ all the way to zero? (One of your reasons should come from Mangan &
Alon's Table 2, not from the simulation.)

In [ ]:
grader.check("q3b")

---
## Question 4 — The ring, and the criterion

*T27, T28. Session 11.*

Three repressors in a ring, each shut off by the one behind it:

$$\frac{dx_i}{dt} = \frac{\alpha}{1+x_{i-1}^{\,n}} - x_i$$

**Q4a.** Write `symmetric_point(alpha, n)` — the level $x$ at which all
three genes sit, from $x(1+x^n) = \alpha$, found with `brentq` — and
`ring_gain(alpha, n)`, the gain around the loop at that point,

$$g = \frac{n\,x^{n}}{1+x^{n}}.$$

**Bracket the root before you write code:** the left side is $0$ at $x=0$ and
exceeds $\alpha$ at $x = \alpha$ (for $\alpha > 0$).

In [ ]:
def symmetric_point(alpha, n):
    """The level every gene sits at, from x (1 + x**n) = alpha."""
    ...


def ring_gain(alpha, n):
    """g = n x**n / (1 + x**n) at the symmetric point."""
    ...


for a in (2.0, 3.7798, 10.0):
    print(f"alpha = {a:7.4f}:  x = {symmetric_point(a, 3):.4f}, g = {ring_gain(a, 3):.4f}")

In [ ]:
grader.check("q4a")

**Q4b.** Write `oscillates(alpha, n)` returning `True` when the symmetric
state cannot hold — **using the criterion**, not by simulating and looking.

Then write `alpha_critical(n)`, the smallest $\alpha$ that oscillates, from
$g = 2$:

$$x^{n} = \frac{2}{n-2}, \qquad \alpha_c = x\left(1+x^{n}\right)$$

and return `np.inf` when $n \le 2$.

In [ ]:
def oscillates(alpha, n):
    """True when the symmetric state is unstable -- i.e. when g > 2."""
    ...


def alpha_critical(n):
    """Smallest alpha that oscillates; infinite for n <= 2."""
    ...


for n in (2, 3, 4, 5):
    print(f"n = {n}: alpha_c = {alpha_critical(n)}")

In [ ]:
grader.check("q4b")

---
## Question 5 — Find the boundary without the algebra

*T29. Session 11.*

Q4's criterion needed the ring to be symmetric. Most circuits are not, and then
there is no closed form. What survives is the procedure: pick a value of the
parameter, find the fixed point, build the Jacobian, take the largest real part
of its eigenvalues, and see where that crosses zero.

**Write the loop yourself first.** `posb` has `sweep`, `leading_real_part` and
`hopf_boundary`, and you may not import them until Q5c.

**Q5a.** Write `max_real_part(alpha, n)`: build the ring with
`posb.repressilator_model`, find its symmetric fixed point with
`posb.fixed_points` (one guess is enough — you know roughly where it is from
Q4a), build the Jacobian with `posb.jacobian`, and return the largest real part
of its eigenvalues.

In [ ]:
def max_real_part(alpha, n):
    """Largest Re(lambda) of the Jacobian at the ring's symmetric fixed point."""
    ...


for a in (2.0, 3.78, 6.0):
    print(f"alpha = {a}: max Re lambda = {max_real_part(a, 3):+.4f}")

In [ ]:
grader.check("q5a")

**Q5b.** Write `find_boundary(n, lo, hi, tol=1e-8)`: your own bisection on
`max_real_part`, returning the $\alpha$ at which it crosses zero. Raise a
`ValueError` if `lo` and `hi` do not straddle the crossing — an interval that
does not bracket a boundary has no answer in it, and returning one anyway is
worse than failing.

Then plot `max_real_part` against $\alpha$ over a range that contains the
crossing, with your answer marked.

In [ ]:
def find_boundary(n, lo, hi, tol=1e-8):
    """Bisect for the alpha where max Re(lambda) crosses zero."""
    ...


found = find_boundary(3, 1.0, 12.0)
print(f"sweep: {found:.6f}   analytic: {alpha_critical(3):.6f}")

aa = np.linspace(1.0, 12.0, 120)
plt.figure(figsize=(5, 3.5))
plt.axhline(0, lw=1, color="0.6")
plt.plot(aa, [max_real_part(a, 3) for a in aa])
plt.axvline(found, ls="--", color="C3")
plt.xlabel(r"$\alpha$"); plt.ylabel(r"max Re $\lambda$");

In [ ]:
grader.check("q5b")

<!-- BEGIN QUESTION -->

**Q5c.** Now import `hopf_boundary` from `posb` and check your answer
against it for $n = 3, 4, 5$.

Then, in two or three sentences: Q4b gave you a closed form, and it is exact.
Why would anyone run Q5b's sweep instead? Name one specific circuit from this
course for which the closed form would not be available.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 6 — required for BioE 247, extra credit for BioE 147

**BioE 247** — part of the assignment, worth 6 of your 48
points.

**BioE 147** — optional, worth up to 6 points of extra credit on top of
42. You do not need it for full marks. It is not a harder version of the
same thing; it removes an assumption the earlier questions made, which is
where most of the interest is.

<!-- BEGIN QUESTION -->

**Q6.** Session 11 derived the criterion for a ring of **three**. Remove
that assumption.

For a ring of $N$ repressors the Jacobian at the symmetric point is still
$J = -I - gP$, but now $P$ is the $N$-cycle, so its eigenvalues are the $N$th
roots of unity $\omega_k = e^{2\pi i k/N}$.

1. Write down $\operatorname{Re}\lambda_k$ for general $N$, and from it a
   formula for the critical gain $g_{\text{crit}}(N)$. Write
   `ring_gain_critical(N)` implementing it, and report the value for
   $N = 3, 4, 5, 6, 7$.
2. Two of those five are qualitatively different from the other three. Say
   which, what is different about the eigenvalue that crosses, and what the
   circuit does instead of oscillating. Connect it to the argument the room
   made in the first ten minutes of session 11.
3. A ring of five needs *less* gain than a ring of three. Explain why in terms
   of what one trip around the loop costs, and say what that implies for the
   relationship between **delay** and oscillation — which is the claim session
   11 closed with.

Put your code below the written answer, and keep the writing to 300 words.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

